# BDW Kaggle Backend — 7B Q4_K_M + Cloudflare Tunnel

Runs the private BDW 7B model on a free Kaggle GPU and exposes it through the
`berojgardegreewala` Cloudflare Tunnel.

## Before running

1. Create a Kaggle notebook and enable **GPU** (T4 or P100) and **Internet**.
2. Add these Kaggle Secrets (Add-ons > Secrets):
   - `HF_TOKEN` — Hugging Face read token for the private model repo
   - `CF_TUNNEL_TOKEN` — Cloudflare Tunnel token
   - `BDW_API_KEY` — any strong random bearer token; the frontend must use the same value
3. Run all cells in order.
4. Keep the notebook session running. Kaggle free sessions last up to 12 hours.


## 1. Install CUDA llama.cpp and serving dependencies

If the prebuilt CUDA wheel is unavailable for the current Kaggle image, use the
source-build fallback in the next markdown cell.


In [ ]:
%pip install -q --upgrade llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
%pip install -q --upgrade fastapi uvicorn huggingface_hub requests


### Fallback if the CUDA wheel install fails

```bash
CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --no-cache-dir --force-reinstall
```


## 2. Load secrets and configuration

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()


def require_secret(name: str) -> str:
    try:
        value = _secrets.get_secret(name)
    except Exception:
        value = os.environ.get(name, "")
    if not value:
        raise RuntimeError(f"Missing Kaggle secret: {name}")
    return value


HF_TOKEN = require_secret("HF_TOKEN")
CF_TUNNEL_TOKEN = require_secret("CF_TUNNEL_TOKEN")
BDW_API_KEY = require_secret("BDW_API_KEY")

MODEL_REPO = os.environ.get("MODEL_REPO", "aamitkr26/bdw-7b-sft")
MODEL_FILE = os.environ.get("MODEL_FILE", "bdw-7b-sft-Q4_K_M.gguf")
PUBLIC_MODEL_ID = os.environ.get("PUBLIC_MODEL_ID", "bdw-7b-sft")
MODEL_DIR = "/kaggle/working/models"
PUBLIC_URLS = [
    "https://ai.berojgardegreewala.app",
    "https://kaggle.berojgardegreewala.app",
]

print("HF token loaded:", bool(HF_TOKEN))
print("Tunnel token loaded:", bool(CF_TUNNEL_TOKEN))
print("API key loaded:", bool(BDW_API_KEY))
print("Model repo:", MODEL_REPO)


## 3. Verify GPU and install cloudflared

In [ ]:
!nvidia-smi


In [ ]:
import os
import stat
import urllib.request

CF_BIN = "/usr/local/bin/cloudflared"
try:
    if not os.path.exists(CF_BIN):
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            CF_BIN,
        )
        os.chmod(CF_BIN, os.stat(CF_BIN).st_mode | stat.S_IEXEC)
except PermissionError:
    CF_BIN = "/kaggle/working/cloudflared"
    if not os.path.exists(CF_BIN):
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            CF_BIN,
        )
        os.chmod(CF_BIN, 0o755)

print("cloudflared:", CF_BIN)


## 4. Download the private GGUF from Hugging Face

In [ ]:
import os

from huggingface_hub import hf_hub_download

os.makedirs(MODEL_DIR, exist_ok=True)
model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir=MODEL_DIR,
    token=HF_TOKEN,
)
print("model:", model_path)
print("size GB:", round(os.path.getsize(model_path) / (1024 ** 3), 2))


## 5. Start the OpenAI-compatible server

The server intentionally requires a bearer token. The frontend gateway sends
`KAGGLE_AI_API_KEY`; it must match `BDW_API_KEY` from the Kaggle secret.


In [ ]:
%%writefile /kaggle/working/server.py
import os
import time

from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.responses import JSONResponse
from llama_cpp import Llama

MODEL_PATH = os.environ["MODEL_PATH"]
MODEL_ID = os.environ.get("PUBLIC_MODEL_ID", "bdw-7b-sft")
API_KEY = os.environ.get("BDW_API_KEY", "")
N_CTX = int(os.environ.get("N_CTX", "8192"))
N_GPU_LAYERS = int(os.environ.get("N_GPU_LAYERS", "-1"))

llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    n_batch=512,
    n_threads=4,
    chat_format="chatml",
    verbose=False,
)

app = FastAPI(title="BDW Kaggle 7B")


def require_auth(authorization: str | None) -> None:
    if API_KEY and authorization != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "context": N_CTX}


@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "created": int(time.time()), "owned_by": "bdw"}]}


@app.post("/v1/chat/completions")
async def chat(request: Request, authorization: str | None = Header(default=None)):
    require_auth(authorization)
    body = await request.json()
    messages = body.get("messages") or []
    max_tokens = min(int(body.get("max_tokens", 1024)), 2048)
    temperature = float(body.get("temperature", 0.3))
    result = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=float(body.get("top_p", 0.9)),
    )
    result["id"] = result.get("id", f"chatcmpl-{int(time.time())}")
    result["object"] = "chat.completion"
    result["model"] = MODEL_ID
    return JSONResponse(result)


In [ ]:
import os
import subprocess
import sys
import time

import requests

env = os.environ.copy()
env.update({
    "MODEL_PATH": model_path,
    "PUBLIC_MODEL_ID": PUBLIC_MODEL_ID,
    "BDW_API_KEY": BDW_API_KEY,
    "N_CTX": "8192",
    "N_GPU_LAYERS": "-1",
})

server_log = open("/kaggle/working/server.log", "w")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/kaggle/working",
    env=env,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)
print("server pid:", server_proc.pid)

ready = False
for _ in range(120):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.ok:
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)

print("server ready:", ready)
if not ready:
    print(open("/kaggle/working/server.log", encoding="utf-8", errors="replace").read()[-4000:])
    raise RuntimeError("llama.cpp server did not become ready")

r = requests.post(
    "http://127.0.0.1:8000/v1/chat/completions",
    headers={"Authorization": f"Bearer {BDW_API_KEY}"},
    json={
        "model": PUBLIC_MODEL_ID,
        "messages": [{"role": "user", "content": "Reply with exactly: BDW ready"}],
        "max_tokens": 16,
        "temperature": 0.1,
    },
    timeout=120,
)
print("test status:", r.status_code)
print(r.text[:500])


## 6. Start the Cloudflare Tunnel

The tunnel ingress is already configured for:

- `ai.berojgardegreewala.app` -> `http://localhost:8000`
- `kaggle.berojgardegreewala.app` -> `http://localhost:8000`

The hostname must have a proxied Cloudflare DNS CNAME pointing to
`edcb48de-dc42-4097-8d5a-d91320250ee2.cfargotunnel.com`.


In [ ]:
import subprocess
import time

cf_log = open("/kaggle/working/cloudflared.log", "w")
cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--protocol", "http2", "--no-autoupdate", "run", "--token", CF_TUNNEL_TOKEN],
    stdout=cf_log,
    stderr=subprocess.STDOUT,
)
print("cloudflared pid:", cf_proc.pid)
time.sleep(15)
print(open("/kaggle/working/cloudflared.log", encoding="utf-8", errors="replace").read()[-3000:])
print("Test one of:", [u + "/health" for u in PUBLIC_URLS])


## 7. Frontend environment variables

Set these in the Vercel project for `www.berojgardegreewala.app`:

```text
KAGGLE_AI_BASE_URL=https://ai.berojgardegreewala.app/v1
KAGGLE_AI_MODEL=bdw-7b-sft
KAGGLE_AI_API_KEY=<same value as the Kaggle BDW_API_KEY secret>
KAGGLE_AI_TIMEOUT_MS=120000
```

Then redeploy the frontend. In `/ask-ai`, select **BDW Sovereign 7B (Kaggle)**.


## 8. Keep the session alive

Run this cell last. Stop the notebook kernel when you are done.


In [ ]:
import time

while True:
    time.sleep(300)
    print("alive", time.strftime("%Y-%m-%d %H:%M:%S"), flush=True)
